# Proyecto Sprint 8 - Análisis de Viajes en Taxi en Chicago para Zuber

## 1. Introducción

La finalidad de este proyecto es identificar patrones clave que ayuden a comprender las **preferencias de los pasajeros** y evaluar el **impacto de factores externos** —como el clima— en los viajes realizados por servicios de taxi en Chicago. Esta información permitirá facilitar la **entrada estratégica de Zuber**, una nueva empresa de viajes compartidos, al mercado local.

Además, se analizarán los datos de los competidores y se pondrá a prueba una hipótesis estadística relacionada con la duración de los viajes en función de las condiciones climáticas.

### Descripción de los datos

**Base de datos principal:**

- `neighborhoods` — Datos sobre los barrios de la ciudad:

    - `name`: nombre del barrio
    - `neighborhood_id`: código del barrio

- `cabs` — Datos sobre los taxis:

    - `cab_id`: código del vehículo
    - `vehicle_id`: ID técnico del vehículo
    - `company_name`: nombre de la empresa propietaria del vehículo

- `trips` — Datos sobre los viajes:

    - `trip_id`: identificador único del viaje
    - `cab_id`: código del vehículo que opera el viaje
    - `start_ts`: fecha y hora de inicio del viaje (redondeado a la hora)
    - `end_ts`: fecha y hora de finalización del viaje (redondeado a la hora)
    - `duration_seconds`: duración del viaje en segundos
    - `distance_miles`: distancia del viaje en millas
    - `pickup_location_id`: código del barrio de recogida
    - `dropoff_location_id`: código del barrio de destino

- `weather_records` — Datos sobre el clima:

    - `record_id`: identificador del registro meteorológico
    - `ts`: fecha y hora del registro (redondeado a la hora)
    - `temperature`: temperatura registrada (en grados Kelvin)
    - `description`: breve descripción del clima (por ejemplo, "light rain", "scattered clouds")

---

**Archivos adicionales obtenidos vía SQL:**

- `project_sql_result_01.csv` — Número de viajes por empresa de taxis (15 y 16 de noviembre de 2017):

    - `company_name`
    - `trips_amount`

- `project_sql_result_04.csv` — Promedio de viajes finalizados por barrio durante noviembre de 2017:

    - `dropoff_location_name`
    - `average_trips`

- `project_sql_result_07.csv` — Datos de viajes del Loop al aeropuerto O'Hare (necesarios para la prueba de hipótesis):

    - `start_ts`
    - `weather_conditions`
    - `duration_seconds`


## 2. Paso 1: Extracción de Datos Meteorológicos

In [1]:
# Importar librerías
import requests # Importa la librería para enviar solicitudes al servidor
from bs4 import BeautifulSoup # Importa la librería para analizar la página web
import pandas as pd # Importa la librería pandas con el alias "pd"
import plotly.express as px
from scipy import stats


In [2]:
# Almancenar la dirección del enlace en la variable "URL"
URL = 'https://practicum-content.s3.us-west-1.amazonaws.com/data-analyst-eng/moved_chicago_weather_2017.html'

# Guardar el objeto de respuesta en la variable requerida "req"
req = requests.get(URL) # Solicitud GET

# Crea un objeto BeautifulSoup y pasa el contenido de texto 
# de la solicitud GET y lo almacena en la variable "soup"
soup = BeautifulSoup(req.text, 'lxml')

# Aplica el método de búsqueda a la etiqueta de la tabla
# especificando el atributo de la tabla "weather_records"
# y lo almacena en la variable "table"
table = soup.find('table', attrs={'id': 'weather_records'})

# Crea un bucle para obtener los encabezados de la tabla 
# y los almacena en la lista "table_headers"
table_headers = []
for row in table.find_all('th'):
    table_headers.append(row.text)

# Crea un bucle para obtener los datos de la tabla 
# y los almacena en la lista "content"
content = []
for row in table.find('tbody').find_all('tr'):
    td_elements = row.find_all('td')
    if td_elements:
        content.append([td.text for td in td_elements])
 
# Crea un DataFrame de pandas llamado "weather_records"
weather_records = pd.DataFrame(content, columns=table_headers)

# Imprime el DataFrame
weather_records.head()

,Date and time,Temperature,Description
0,2017-11-01 00:00:00,276.150,broken clouds
1,2017-11-01 01:00:00,275.700,scattered clouds
2,2017-11-01 02:00:00,275.610,overcast clouds
3,2017-11-01 03:00:00,275.350,broken clouds
4,2017-11-01 04:00:00,275.240,broken clouds


## 2.2 Análisis exploratorio de datos meteorológicos

En esta seccion, se explorarán los datos del DataFrame usando los siguientes métodos para obtener una comprensión inicial:

- `sample()` para visualizar una muestra de los valores y su descripción.
- `info()` para conocer el tamaño del DataFrame, ver los tipos de datos e identificar valores nulos.
- `describe()` para obtener estadísticas generales.


Nos enfocaremos en identificar posibles problemas como:

- Valores faltantes.
- Valores duplicados.
- Tipos de datos incorrectos.
- Inconsistencias en los datos (uso inconsistente de mayúsculas, espacios innecesarios en texto, etc.).


Una vez inspeccionados los datos, se realizarán las correcciones necesarias para garantizar la calidad de los mismos antes del análisis.

In [3]:
# Vista previa de los datos:
print('Muestra aleatoria de los datos:')
weather_records.sample(10)

Muestra aleatoria de los datos:


,Date and time,Temperature,Description
661,2017-11-28 13:00:00,283.960,sky is clear
102,2017-11-05 06:00:00,283.230,fog
196,2017-11-09 04:00:00,273.060,mist
479,2017-11-20 23:00:00,283.860,scattered clouds
259,2017-11-11 19:00:00,273.530,overcast clouds
454,2017-11-19 22:00:00,274.690,broken clouds
150,2017-11-07 06:00:00,277.400,overcast clouds
580,2017-11-25 04:00:00,284.610,scattered clouds
400,2017-11-17 16:00:00,276.320,haze
207,2017-11-09 15:00:00,277.590,overcast clouds


In [4]:
# Imprime la información general/resumida sobre el DataFrame "weather_records"
print('Información general del Dataframe')
print()
print(weather_records.info())
print()
print()
print(f'Número de filas duplicadas: {weather_records.duplicated().sum()}')


Información general del Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Date and time  697 non-null    object
 1   Temperature    697 non-null    object
 2   Description    697 non-null    object
dtypes: object(3)
memory usage: 16.5+ KB
None


Número de filas duplicadas: 0


### Observaciones del análisis preliminar

- El DataFrame `weather_records` tiene 697 registros y 3 columnas.
- No hay valores nulos ni filas duplicadas.
- Sin embargo, todas las columnas están en formato `object`, incluyendo la temperatura y la marca temporal.
- Las estadísticas descriptivas no son útiles mientras los datos estén en este formato.
- Es necesario convertir:
  - `Date and time` a tipo `datetime`
  - `Temperature` a tipo `float`
- Normalizaremos de nombres de las columnas:
    - Convertiremos todos los nombres de columnas a minúsculas.
    - Cambiaremos el nombre de la columna `Date and time` a `ts` para que coincida con las instrucciones
- También, aunque no se detectó ninguna anomalía al revisar la muestra aleatoria, se recomienda estandarizar la columna `Description` para detectar y prevenir duplicados implícitos (por ejemplo, "Rain" vs "rain", o espacios adicionales).

In [5]:
# Renombra las columnas para que estén todas en minúsculas y seguir las buenas prácticas
weather_records.columns = ['ts', 'temperature', 'description']

# Elimina espacios extras y convierte a minúsculas los valores en la columna "description"
weather_records['description'] = weather_records['description'].str.strip().str.lower()

# Convierte 'timestamp' a formato "datetime"
weather_records['ts'] = pd.to_datetime(weather_records['ts'])

# Convierte 'temperature' a "float"
weather_records['temperature'] = pd.to_numeric(weather_records['temperature'], errors='coerce')

In [6]:
# Verificar tipos de datos y valores nulos
print('Información general del Dataframe después de la corrección de los tipos de datos')
print()
weather_records.info()

Información general del Dataframe después de la corrección de los tipos de datos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ts           697 non-null    datetime64[ns]
 1   temperature  697 non-null    float64       
 2   description  697 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 16.5+ KB


In [7]:
# Verifica los nombres de las columnas después de haberlas normalizado
print('Nombres de las columnas después de la transformación:')
weather_records.columns

Nombres de las columnas después de la transformación:


Index(['ts', 'temperature', 'description'], dtype='object')

In [8]:
# Estadísticas descriptivas
print('Estadísticas descriptivas de la temperatura')
print()
print(weather_records['temperature'].describe())
print()
print()
# Número de valores únicos en la columna 'description'
print(f'Número de valores únicos en la columna description: {weather_records['description'].value_counts()}')
print()
print()
# Mostrar todos los valores únicos
print('Nombre de todos los valores únicos en la descripción del clima:')
print()
weather_records['description'].unique()


Estadísticas descriptivas de la temperatura

count    697.000000
mean     277.548864
std        4.515403
min      265.740000
25%      274.240000
50%      277.410000
75%      280.980000
max      289.760000
Name: temperature, dtype: float64


Número de valores únicos en la columna description: description
sky is clear                        178
overcast clouds                     146
mist                                 97
broken clouds                        63
scattered clouds                     47
few clouds                           37
light rain                           31
fog                                  24
haze                                 18
light intensity drizzle              13
moderate rain                        12
light snow                           11
drizzle                               9
proximity thunderstorm                5
proximity thunderstorm with rain      2
thunderstorm with drizzle             1
thunderstorm with light rain          1
heavy intensity

array(['broken clouds', 'scattered clouds', 'overcast clouds',
       'sky is clear', 'mist', 'drizzle', 'light rain', 'moderate rain',
       'fog', 'light intensity drizzle', 'few clouds',
       'thunderstorm with drizzle', 'proximity thunderstorm',
       'proximity thunderstorm with rain', 'light snow', 'haze',
       'thunderstorm with light rain', 'heavy intensity rain',
       'thunderstorm with rain'], dtype=object)

### Análisis descriptivo de la temperatura y descripción del clima

- El DataFrame tiene 697 registros con temperaturas registradas en lo que parece ser grados **Kelvin**.
- La conversión de los extremos da como resultado:
  - Mínima: 265.74 K — -7.41 ºC
  - Máxima: 289.76 K — 16.61 ºC
- Esto concuerda con un clima otoñal en Chicago, donde es común experimentar tanto temperaturas bajo cero como temperaturas templadas en un mismo mes.
- La **media** y la **mediana** son muy similares (~277.5 K o 4.26 ºC), lo cual sugiere una distribución bastante simétrica o normal.
- A pesar de que el rango total entre el valor mínimo y el máximo abarca 22 K (Kelvin), la mayoría de los valores parecen estar concentrados cerca del promedio, lo que podría sugerir que las condiciones climáticas son relativamente estables dentro del mes.
- La descripción del clima muestra 19 categorías únicas. Algunas observaciones:
  - Las más frecuentes son `'sky is clear'`, `'overcast clouds'`, `'mist'`, `'broken clouds'`, `'scattered clouds'` y `'few clouds'` y `'light rain  '`.
  - Se identifican múltiples tipos de precipitación: `'light rain'`, `'moderate rain'`, `'drizzle'`, `'thunderstorm'`, `'light snow'` y `'heavy intensity rain'`.
  - No hay duplicados implícitos ni inconsistencias en la categorización.

Con base en estos resultados, el siguiente paso será clasificar las condiciones climáticas en dos grupos: **"Good"** y **"Bad"** cuando realicemos el análisis exploratorio con SQL para facilitar el análisis de su posible relación con la duración de los viajes.


## 3. Paso 2: Análisis Exploratorio con SQL

Este bloque contiene las principales consultas SQL realizadas para comprender la actividad de los viajes:

- **Consulta 1**: Se recuperó el número total de viajes por empresa de taxis durante los días 15 y 16 de noviembre. Se usó `JOIN` entre `cabs` y `trips`, con filtrado por fecha (`DATE(trips.start_ts)`), agregación con `COUNT()`, y agrupamiento por nombre de la empresa.

- **Consulta 2**: Se filtraron los viajes de empresas que contienen las palabras "Yellow" o "Blue" en su nombre durante el periodo del 1 al 7 de noviembre. Se aplicó `LIKE` para detectar patrones en los nombres (por medio de expresiones regulares), junto con agrupación y conteo.

- **Consulta 3**: Se compararon las dos compañías más populares con todas las demás agrupadas como "Other". Se utilizó `CASE` dentro de `SELECT` para etiquetar las empresas y luego se agruparon los resultados para visualización.

**Observaciones**:
- Flash Cab y Taxi Affiliation Services son las más activas en el periodo analizado.
- Las consultas preparan los datos que luego serán visualizados en gráficos para evaluar tendencias.

## 4. Paso 3: Preparación para Prueba de Hipótesis

Este bloque muestra los pasos para preparar los datos requeridos para la prueba de hipótesis:

- **Identificación de barrios**: Se identificaron los `neighborhood_id` para `Loop` y `O'Hare` con una consulta a `neighborhoods` usando `LIKE` y `=`.

- **Clasificación del clima**: Se aplicó `CASE` sobre la tabla `weather_records` para categorizar las condiciones meteorológicas en "Good" o "Bad" basadas en la aparición de las palabras "rain" o "storm".

- **Filtrado de viajes relevantes**: Se seleccionaron solo los viajes que comenzaron en `Loop` y finalizaron en `O'Hare`, ocurridos un sábado, utilizando `EXTRACT(DOW FROM start_ts) = 6`.

- **Unión con condiciones meteorológicas**: Finalmente, se hizo un `JOIN` entre `trips` y `weather_records` por fecha y hora (`start_ts = ts`) para unir cada viaje con su respectiva condición climática.

**Observaciones**:
- Los datos resultantes permiten analizar si la duración de los viajes varía con base en condiciones meteorológicas—"Good" y "Bad".
- Este conjunto fue exportado automáticamente como el archivo `project_sql_result_07.csv` para su análisis en Python.

## 5. Paso 4: Análisis en Python

### 5.1 Carga y limpieza de `project_sql_result_01.csv`

- Contiene el número de viajes por empresa (15 y 16 de noviembre).
- Nos aseguraremos de que no haya valores nulos y que los tipos de datos sean correctos.


In [9]:
# Carga los datos
df_company_trips = pd.read_csv('moved_project_sql_result_01.csv')

# Imprime la información general/resumida sobre el DataFrame "df_company_trips"
print('Información general del Dataframe')
print()
print(df_company_trips.info())
print()
print()
# Número de filas duplicadas
print(f'Número de filas duplicadas: {df_company_trips.duplicated().sum()}')
print()
print()
# Vista previa de los datos:
print('Muestra aleatoria de los datos:')
df_company_trips.sample(10)


Información general del Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   company_name  64 non-null     object
 1   trips_amount  64 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.1+ KB
None


Número de filas duplicadas: 0


Muestra aleatoria de los datos:


,company_name,trips_amount
34,Norshore Cab,127
11,Globe Taxi,4383
3,Yellow Cab,9888
25,Top Cab Affiliation,978
22,Chicago Independents,1296
31,American United Taxi Affiliation,210
60,2733 - 74600 Benny Jona,7
36,3591 - 63480 Chuks Cab,37
10,Choice Taxi Association,5015
50,2823 - 73307 Lee Express Inc,15


In [10]:
# Mostrar todos los valores únicos
print(f'Total de valores únicos en la columna "company_name": {df_company_trips['company_name'].nunique()}')
print()
print()
print('Nombre de todos los valores únicos:')
print()
print(df_company_trips['company_name'].unique())
print()
print()
# Estadísticas descriptivas
print('Estadísticas descriptivas de la cantidad de viajes')
print()
print(df_company_trips['trips_amount'].describe())


Total de valores únicos en la columna "company_name": 64


Nombre de todos los valores únicos:

['Flash Cab' 'Taxi Affiliation Services' 'Medallion Leasin' 'Yellow Cab'
 'Taxi Affiliation Service Yellow' 'Chicago Carriage Cab Corp'
 'City Service' 'Sun Taxi' 'Star North Management LLC'
 'Blue Ribbon Taxi Association Inc.' 'Choice Taxi Association'
 'Globe Taxi' 'Dispatch Taxi Affiliation' 'Nova Taxi Affiliation Llc'
 'Patriot Taxi Dba Peace Taxi Associat' 'Checker Taxi Affiliation'
 'Blue Diamond' 'Chicago Medallion Management' '24 Seven Taxi'
 'Chicago Medallion Leasing INC' 'Checker Taxi' 'American United'
 'Chicago Independents' 'KOAM Taxi Association' 'Chicago Taxicab'
 'Top Cab Affiliation' 'Gold Coast Taxi' 'Service Taxi Association'
 '5 Star Taxi' '303 Taxi' 'Setare Inc' 'American United Taxi Affiliation'
 'Leonard Cab Co' 'Metro Jet Taxi A' 'Norshore Cab'
 '6742 - 83735 Tasha ride inc' '3591 - 63480 Chuks Cab'
 '1469 - 64126 Omar Jada' '6743 - 78771 Luhak Corp'
 '0118 - 42111 G

#### Conclusiones del Análisis Exploratorio de `project_sql_result_01.csv`

- El DataFrame tiene 64 registros únicos, sin valores nulos ni duplicados, y con tipos de datos correctos.

- El número de viajes por empresa presenta una enorme dispersión:
    - La empresa más activa tuvo **19,558 viajes**, mientras que la menos activa solo **2**.
    - La **media** fue de **2,145**, pero la **mediana** fue de apenas **178**, lo cual sugiere que hay valores extremadamente altos que elevan la media.
    - La **desviación estándar** de **3,812** confirma esta variabilidad.

- La distribución no es homogénea: unas pocas empresas concentran la mayor parte de los viajes, mientras que muchas otras tienen una participación muy baja.

- Esto justifica que el gráfico de barras (sección 5.3) sea el recurso más apropiado, ya que permitirá visualizar de forma efectiva la distribución desigual.


### 5.2 Carga y limpieza de `project_sql_result_04.csv`

- Contiene el número promedio de viajes que finalizaron en cada barrio.
- Se analizarán los 10 barrios más frecuentes.

In [11]:
# Carga los datos
df_top_locations = pd.read_csv('moved_project_sql_result_04.csv')

# Imprime la información general/resumida sobre el DataFrame "df_top_locations"
print('Información general del Dataframe')
print()
print(df_top_locations.info())
print()
print()
# Número de filas duplicadas
print(f'Número de filas duplicadas: {df_top_locations.duplicated().sum()}')
print()
print()
# Vista previa de los datos:
print('Muestra aleatoria de los datos:')
df_top_locations.sample(10)

Información general del Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 2 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   dropoff_location_name  94 non-null     object 
 1   average_trips          94 non-null     float64
dtypes: float64(1), object(1)
memory usage: 1.6+ KB
None


Número de filas duplicadas: 0


Muestra aleatoria de los datos:


,dropoff_location_name,average_trips
72,South Chicago,13.000000
18,West Town,516.400000
36,Kenwood,89.366667
26,Hyde Park,255.133333
56,Ukrainian Village,27.600000
7,Museum Campus,1510.000000
43,Mckinley Park,64.033333
70,Wrigleyville,14.233333
10,Lincoln Park,1246.533333
5,Lake View,2420.966667


In [12]:
# Mostrar todos los valores únicos
print(f'Total de valores únicos en la columna "dropoff_location_name": {df_top_locations['dropoff_location_name'].nunique()}')
print()
print()
print('Nombre de todos los valores únicos:')
print()
print(df_top_locations['dropoff_location_name'].unique())
print()
print()
# Estadísticas descriptivas
print('Estadísticas descriptivas de la cantidad de viajes')
print()
print(df_top_locations['average_trips'].describe())

Total de valores únicos en la columna "dropoff_location_name": 94


Nombre de todos los valores únicos:

['Loop' 'River North' 'Streeterville' 'West Loop' "O'Hare" 'Lake View'
 'Grant Park' 'Museum Campus' 'Gold Coast' 'Sheffield & DePaul'
 'Lincoln Park' 'East Village' 'Little Italy, UIC' 'Uptown'
 'Near South Side' 'Garfield Ridge' 'Logan Square' 'Edgewater' 'West Town'
 'Old Town' 'Rush & Division' 'North Center' 'Lincoln Square'
 'Rogers Park' 'West Ridge' 'Irving Park' 'Hyde Park' 'Avondale'
 'Wicker Park' 'Albany Park' 'United Center' 'Lower West Side' 'Douglas'
 'Portage Park' 'Humboldt Park' 'Norwood Park' 'Kenwood' 'Bridgeport'
 'Armour Square' 'Jefferson Park' 'Bucktown' 'North Park' 'Garfield Park'
 'Mckinley Park' 'Belmont Cragin' 'Boystown' 'Chinatown' 'Grand Boulevard'
 'Austin' 'Sauganash,Forest Glen' 'South Shore' 'Woodlawn'
 'Little Village' 'Jackson Park' 'North Lawndale' 'Dunning'
 'Ukrainian Village' 'Hermosa' 'Englewood' 'Chatham' 'New City'
 'Grand Crossing' 'Brig

#### Promedio de viajes por barrio `project_sql_result_04.csv`

- Este DataFrame contiene 94 barrios únicos, sin valores ausentes ni duplicados,  y con tipos de datos correctos.

- Se observa una **alta variabilidad en el promedio de viajes por barrio**:

    - El barrio con más viajes completados tuvo un promedio de **10,727**, mientras que el menor fue de **1.8**.

    - La **media** fue de **599.9**, mientras que la **mediana** fue apenas **52.01**.

    - La **desviación estándar** fue de **1,714**, indicando una distribución fuertemente sesgada hacia unos pocos barrios muy transitados.

- Los datos muestran que algunos barrios son puntos clave para el descenso de pasajeros, mientras que otros reciben muy poca actividad.

- Por esta razón, también se usará un gráfico de barras para los 10 principales barrios, permitiendo comparar claramente las zonas más relevantes.

### 5.3 Gráficos

- Se visualizarán los datos de las tablas anteriores.
- Se comentarán tendencias observadas.

In [13]:
# Gráfico 1: 10 Empresas principales vs número de viajes
df_top10_companies = df_company_trips.sort_values(by='trips_amount', ascending=False).head(10)
fig1 = px.bar(df_top10_companies,
              x='company_name', y='trips_amount',
              title='Número de viajes por empresa (15-16 nov)',
              labels={'company_name': 'Empresa', 'trips_amount': 'Viajes'},
              text='trips_amount'
              )
# Configurar la cuadrícula en el eje y
fig1.update_yaxes(showgrid=True, gridcolor='lightblue', gridwidth=1, griddash='dot')

fig1.show()

<div class="alert alert-block alert-info">
<b>Hola, Tonatiuh, muchas gracias por tus comentarios. He realizado la corrección de la primera gráfica para que solo se muestren las 10 empresas pricipales. Quedo atenta a la siguiente revisión.</b> <a class="tocSkip"></a>
</div>

#### Reflexión posterior al Gráfico 1: Número de viajes por empresa

La visualización presenta una concentración muy pronunciada: unas pocas compañías dominan la actividad de viajes en taxi. En particular, **Flash Cab** sobresale como el líder del mercado, con una ventaja de más del **42%** sobre su competidor más cercano, **Taxi Affiliation Services**.

El resto de empresas tiene una presencia significativamente menor, lo cual refleja un **mercado altamente concentrado**. Aunque no es necesario para el análisis principal del proyecto, podría ser útil generar un gráfico enfocado únicamente en el Top 10 de compañías si se quisiera hacer una exploración complementaria. Esto permitiría identificar posibles referentes para Zuber al ingresar al mercado o seleccionar una muestra representativa para modelos predictivos futuros.

In [14]:
# Gráfico 2: 10 barrios principales por número de finalizaciones
df_top10_locations = df_top_locations.sort_values(by='average_trips', ascending=False).head(10)
fig2 = px.bar(df_top10_locations,
              x='dropoff_location_name',
              y='average_trips',
              title='Top 10 barrios por finalización de viajes',
              labels={'dropoff_location_name': 'Barrio', 'average_trips':'Viajes promedio'},
              text='average_trips',
              text_auto='.2f'
              )

# Configurar la cuadrícula en el eje y
fig2.update_yaxes(showgrid=True, gridcolor='lightblue', gridwidth=1, griddash='dot')

fig2.show()

#### Reflexión posterior al Gráfico 2: Top 10 barrios por finalización de viajes

El gráfico muestra también una distribución muy sesgada. **Loop**, el centro financiero de Chicago, lidera ampliamente, lo cual es comprensible dado su dinamismo económico y comercial. Lo siguen **River North**, **Streeterville** y **West Loop**, zonas conocidas por su actividad empresarial, turística y gastronómica.

En contraste, barrios como **Grant Park** o **Gold Coast** tienen un número de viajes notablemente menor. En el caso de **Gold Coast**, esto podría deberse al perfil socioeconómico alto de sus residentes, quienes posiblemente cuenten con transporte privado. **O’Hare**, aunque es un aeropuerto internacional, también aparece con un promedio bajo, lo cual puede explicarse por su rol principalmente como punto de entrada/salida de la ciudad, más que como destino frecuente de taxis locales.

Estas observaciones podrían ser clave para Zuber, ya que permiten identificar zonas estratégicas para su operación inicial, así como comprender patrones de movilidad urbana en función del perfil del barrio.

## 6. Paso 5: Prueba de Hipótesis en Python

En esta seccion, se explorarán los datos del DataFrame usando los siguientes métodos para obtener una comprensión inicial:

- `head()` para tener una vista previa del DataFrame.
- `info()` para conocer el tamaño del DataFrame, ver los tipos de datos e identificar valores nulos.

Nos enfocaremos en identificar posibles problemas como:

- Valores faltantes.
- Tipos de datos incorrectos.
- Inconsistencias en los datos (uso inconsistente de mayúsculas, espacios innecesarios en texto, etc.).


Una vez inspeccionados los datos, se realizarán las correcciones necesarias para garantizar la calidad de los mismos antes del la prueba de hipótesis para responder la pregunta: *¿La duración promedio de los viajes desde el Loop hasta el Aeropuerto Internacional O'Hare cambia los sábados lluviosos?*

In [15]:
# Carga los datos
df_loop_ohare = pd.read_csv('moved_project_sql_result_07.csv')

# Imprime la información general sobre el DataFrame
print('Información general del Dataframe')
print()
print(df_loop_ohare.info())
print()
print()
# Vista previa de los datos:
print('Vista previa de los datos:')
df_loop_ohare.head()

Información general del Dataframe

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   start_ts            1068 non-null   object 
 1   weather_conditions  1068 non-null   object 
 2   duration_seconds    1068 non-null   float64
dtypes: float64(1), object(2)
memory usage: 25.2+ KB
None


Vista previa de los datos:


,start_ts,weather_conditions,duration_seconds
0,2017-11-25 16:00:00,Good,2410.0
1,2017-11-25 14:00:00,Good,1920.0
2,2017-11-25 12:00:00,Good,1543.0
3,2017-11-04 10:00:00,Good,2512.0
4,2017-11-11 07:00:00,Good,1440.0


#### Observaciones del análisis preliminar

- El DataFrame `df_loop_ohare` tiene 1,068 registros y 3 columnas.
- No hay valores nulos.
- Existen valores duplicados, no obstante, esto puede deberse a que diferentes usuarios realizaron viajes en la misma fecha y dentro de la misma hora. 
- Los tipos de datos son correctos, excepto para `start_ts`, el cual está en formato `object`. Aunque no sera necesario para la prueba de hipótesis, se convertirá al tipo correcto `datetime`.

In [16]:
# Convierte 'start_ts' a formato "datetime"
df_loop_ohare['start_ts'] = pd.to_datetime(df_loop_ohare['start_ts'])

# Verifcar tipos de datos
print('Tipos de datos después de la conversión:')
df_loop_ohare.dtypes


Tipos de datos después de la conversión:


start_ts              datetime64[ns]
weather_conditions            object
duration_seconds             float64
dtype: object

### 6.1 Prueba de hipótesis

A continuación, realizaremos la prueba de hipótesis para responder la pregunta: *¿La duración promedio de los viajes desde el Loop hasta el Aeropuerto Internacional O'Hare cambia los sábados lluviosos?*

Para ello, realizaremos:

1. División del DataFrame en dos grupos:

    - Grupo 1: viajes con clima "Bad"

    - Grupo 2: viajes con clima "Good"

2. Formulación de hipótesis:

    - **H₀ (hipótesis nula):** La duración promedio de los viajes desde el Loop hasta O’Hare **es la misma** durante sábados con buen clima y con mal clima.  
    - **H₁ (hipótesis alternativa):** La duración promedio **cambia** significativamente entre sábados con buen clima y sábados con mal clima.

3. Selección de la prueba estadística 

    Para comparar la duración promedio de los viajes entre sábados con buen y mal clima, evaluaremos primero si las varianzas son homogéneas usando la prueba de **Levene**.

    - Si la prueba de Levene indica que las varianzas son iguales (p > 0.05), utilizaremos la prueba **t de Student** para muestras independientes con varianzas iguales.
    - Si la prueba de Levene indica que las varianzas son diferentes (p ≤ 0.05), utilizaremos la prueba **t de Welch**, que no asume varianzas iguales.

In [17]:
# Dividir los datos en dos grupos según las condiciones climáticas
durations_good = df_loop_ohare[df_loop_ohare['weather_conditions'] == 'Good']['duration_seconds'].dropna()
durations_bad = df_loop_ohare[df_loop_ohare['weather_conditions'] == 'Bad']['duration_seconds'].dropna()

# --- Prueba de Levene para evaluar la igualdad de las varianzas y determinar qué prueba t realizar---
results_levene = stats.levene(durations_good, durations_bad)
print('Estadístico de Levene:', results_levene.statistic)
print('p-value de Levene:', results_levene.pvalue)

alpha = 0.05

if results_levene.pvalue > alpha: 
    print('No hay evidencia suficiente para rechazar la igualdad de varianzas. Usaremos la prueba t de Student.')
else:
    print('Las varianzas son significativamente diferentes. Usaremos la prueba t de Welch.')



Estadístico de Levene: 0.38853489683656073
p-value de Levene: 0.5332038671974493
No hay evidencia suficiente para rechazar la igualdad de varianzas. Usaremos la prueba t de Student.


#### **Resultados de la prueba de Levene**

La prueba de Levene arrojó los siguientes resultados:

- Estadístico: 0.389
- Valor p: 0.533

Dado que el valor p es mayor a 0.05, no tenemos evidencia suficiente para rechazar la hipótesis nula de igualdad de varianzas. Por lo tanto, podemos asumir que las varianzas son iguales y aplicar la prueba t de Student `(equal_var=True)` para comparar las medias.

#### **Prueba t  de Student**

Definiremos el nivel de significancia α = 0.05 como umbral para la toma de decisiones:
- Si el p-valor < 0.05, la diferencia es estadísticamente significativa, por lo que rechazamos la hipótesis nula y concluimos que hay una diferencia significativa en la duración promedio de los viajes entre sábados con buen y mal clima.
- Si el p-valor ≥ 0.05, no hay suficiente evidencia para rechazar la hipótesis nula, es decir, no hay diferencia significativa en la duración promedio de los viajes entre sábados con buen y mal clima.


In [18]:
# --- Aplica la prueba t (Student): ---
alpha = 0.05
results = stats.ttest_ind(durations_good, durations_bad, equal_var=True)

print('\nPrueba T (Sudent):')
print('Estadístico t:', results.statistic)
print('\np-value:', results.pvalue)

# Interpreta el p-valor
if results.pvalue < alpha:
    print('\nHay suficiente evidencia para rechazar la hipótesis nula.')
    print('Conclusión: Hay una diferencia significativa en la duración promedio ' \
    'de los viajes entre sábados con buen y mal clima.')
else:
    print('\nNo hay suficiente evidencia para rechazar la hipótesis nula.')
    print('Conclusión: No se encontró una diferencia significativa en la duración promedio ' \
    'de los viajes entre sábados con buen y mal clima.')



Prueba T (Sudent):
Estadístico t: -6.946177714041499

p-value: 6.517970327099473e-12

Hay suficiente evidencia para rechazar la hipótesis nula.
Conclusión: Hay una diferencia significativa en la duración promedio de los viajes entre sábados con buen y mal clima.


In [19]:
# Crea gráfico de caja para comparar duración de viajes en buen y mal clima
fig4 = px.box(df_loop_ohare,
              x='weather_conditions',
              y='duration_seconds',
              points='all',
              color='weather_conditions',
              title="Distribución de duración de viajes: Loop a O'Hare (sábados)",
              labels={
                  'weather_conditions': 'Condiciones climáticas',
                  'duration_seconds': 'Duración del viaje (segundos)'
              })

# Muestra el gráfico
fig4.show()

El gráfico anterior refuerza visualmente los resultados obtenidos en la prueba estadística. Se observa que los viajes realizados los sábados con mal clima ("Bad") tienen una clara tendencia a durar más tiempo y una distribución más concentrada en comparación con los drealizados con un buen clima ("Good"). Esta diferencia entre ambas varianzas también fue estadísticamente significativa según la prueba t de Student.


### 6.2 Resultados de la prueba de hipótesis

**Hipótesis planteadas**  
- H₀ (hipótesis nula): La duración promedio de los viajes desde el Loop hasta O’Hare **es la misma** durante sábados con buen clima y con mal clima.  
- H₁ (hipótesis alternativa): La duración promedio **cambia** significativamente entre sábados con buen clima y sábados con mal clima.

**Nivel de significancia establecido (α):** 0.05

**1. Prueba de Levene — Evaluación de igualdad de varianzas**  
- Estadístico: 0.389  
- Valor p: 0.533  
*Conclusión:* No se rechaza H₀. Las varianzas pueden considerarse iguales—se puede usar la prueba **t de Student**.

**2. Prueba t de Student**  
- Estadístico t: -6.95  
- Valor p: 6.52 × 10⁻¹²  

**Conclusión final:**  
El valor p es significativamente menor que el nivel de significancia establecido (0.05).  
Por lo tanto, **rechazamos la hipótesis nula**.  
Existe una diferencia estadísticamente significativa en la duración promedio de los viajes desde el Loop hasta O’Hare durante sábados con mal clima en comparación con sábados con buen clima.

## 7. Conclusiones Finales del Proyecto

Durante este análisis, exploramos la información disponible sobre los viajes en taxi en la ciudad de Chicago en noviembre de 2017, con el objetivo de identificar patrones clave y evaluar el impacto del clima en la duración de los viajes. A continuación se resumen los hallazgos más relevantes:

- **Dominio del mercado**: Flash Cab lideró con una amplia ventaja en la cantidad de viajes realizados, superando por más del 40% a su competidor más cercano. Este hallazgo plantea oportunidades estratégicas para nuevos actores como Zuber, al identificar zonas donde la competencia no es tan feroz.

- **Zonas de alta demanda**: El distrito Loop y barrios como River North y Streeterville fueron los destinos más frecuentes. Estas zonas se caracterizan por una alta actividad comercial, turística y corporativa, lo que las convierte en focos ideales para captar usuarios.

- **Variabilidad por ubicación**: La diferencia en el número de viajes por barrio fue significativa. Algunos barrios, como Riverdale o Burnside, mostraron un número considerablemente menor, posiblemente por su perfil demográfico o menor densidad de servicios.

- **Impacto del clima en la duración de viajes**:
    - Se confirmó, mediante una prueba estadística t de Student (tras validar la igualdad de varianzas con Levene), que **la duración promedio de los viajes del Loop al aeropuerto O’Hare es significativamente mayor en días lluviosos**.
    - Este hallazgo destaca la necesidad de estrategias de asignación dinámica de unidades en condiciones climáticas adversas para mantener la eficiencia y satisfacción del cliente.

- **Técnicas utilizadas**: Se aplicaron técnicas de *scraping* con BeautifulSoup, consultas SQL complejas, integración de múltiples fuentes de datos, visualización con Plotly y pruebas de hipótesis en Python. Estas herramientas permitieron un análisis integral, tanto exploratorio como confirmatorio.


---

### Recomendaciones para Zuber:

1. **Focalizar campañas de lanzamiento en zonas de alta demanda**, especialmente el Loop y sus alrededores.
2. **Monitorear en tiempo real las condiciones climáticas** y adaptar la logística de asignación de unidades, especialmente durante el fin de semana.
3. **Estudiar con mayor detalle a los líderes del mercado** para identificar brechas en el servicio y oportunidades de diferenciación.


---

### Comentario final

Este análisis sienta una base sólida para la entrada estratégica de Zuber al mercado de Chicago. Gracias al enfoque estructurado, la limpieza de datos rigurosa y la integración de herramientas profesionales, se logró extraer un valor real a partir de datos complejos y de orígenes diversos.